## Deep Research

In [1]:
from agents import Agent, WebSearchTool, trace, Runner, function_tool
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import asyncio
from IPython.display import display, Markdown
from messenger import send_email, push

In [2]:
load_dotenv(override=True)

True

In [3]:
# constants
MODEL_NAME= "gpt-5.4-mini"
USE_EMAIL= True
HOW_MANY_SEARCHES = 5

## Agent 1: The Search Agent

In [4]:
INSTRUCTIONS = """
You are a research assistant. Given a search term, you search the web for that term and 
produce a concise summary of the results. The summary must 2-3 paragraphs and less than 300 words.
Capture the main points and be succinct. Reply only with the summary.
"""

task = "Most popular AI Agent frameworks in 2026"

settings = ModelSettings(tool_choice="required")
tools = [WebSearchTool()]

In [5]:
search_agent = Agent(name = "Search Agent",instructions=INSTRUCTIONS,tools=tools,model=MODEL_NAME,model_settings=settings)

## Agent 2: The Planner Agent

### We will now use Structured Outputs, and include a description of the fields

In [6]:
class WebSearchItem(BaseModel):
    reason: str = Field(description="Your reasoning for why this search is important to the query.")
    query: str = Field(description="The search term to use for the web search.")

class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="A list of web searches to perform to best answer the query.")

In [7]:
WebSearchPlan.model_json_schema()

{'$defs': {'WebSearchItem': {'properties': {'reason': {'description': 'Your reasoning for why this search is important to the query.',
     'title': 'Reason',
     'type': 'string'},
    'query': {'description': 'The search term to use for the web search.',
     'title': 'Query',
     'type': 'string'}},
   'required': ['reason', 'query'],
   'title': 'WebSearchItem',
   'type': 'object'}},
 'properties': {'searches': {'description': 'A list of web searches to perform to best answer the query.',
   'items': {'$ref': '#/$defs/WebSearchItem'},
   'title': 'Searches',
   'type': 'array'}},
 'required': ['searches'],
 'title': 'WebSearchPlan',
 'type': 'object'}

In [8]:
# See note above about cost of WebSearchTool

INSTRUCTIONS2 = f"""
You are a research assistant. Given a user query, come up with a set of web searches
to perform to best answer the query. Output {HOW_MANY_SEARCHES} terms to query for.
"""

planner_agent= Agent(name="Planner Agent",instructions = INSTRUCTIONS2,model=MODEL_NAME,output_type=WebSearchPlan)

## Agent 3: The Writer Agent

In [9]:
INSTRUCTIONS3 = """
You are a senior researcher tasked with writing a cohesive report for a research query.
You will be provided with the original query, and some research.
Generate a comprehensive report based on the research and the query.
The final output should be in markdown format, and it should be lengthy and detailed. Aim 
for 5-10 pages of content, at least 1000 words.
"""

In [10]:
class ReportData(BaseModel):
    short_summary: str = Field(description="A short 2-3 sentence summary of the findings.")
    markdown_report: str = Field(description="The final report")
    follow_up_questions: list[str] = Field(description="Suggested topics to research further")


writer_agent = Agent(name = "Writer Agent",instructions = INSTRUCTIONS3,model = MODEL_NAME,output_type=ReportData)

## Agent 4: The email agent

In [11]:
@function_tool
def send_email_tool(subject: str, text_body: str, html_body: str) -> str:
    """
    Send out an email with the given subject and body to all sales prospects
    
    Args:
        subject: The subject of the email
        text_body: The body of the email as plain text
        html_body: The HTML body of the email
    """
    if USE_EMAIL:
        send_email(subject, text_body, html_body)
    else:
        push(f"Subject: {subject}\n\n{text_body}")
    return "Email sent successfully"

In [12]:
send_email_tool.params_json_schema

{'properties': {'subject': {'description': 'The subject of the email',
   'title': 'Subject',
   'type': 'string'},
  'text_body': {'description': 'The body of the email as plain text',
   'title': 'Text Body',
   'type': 'string'},
  'html_body': {'description': 'The HTML body of the email',
   'title': 'Html Body',
   'type': 'string'}},
 'required': ['subject', 'text_body', 'html_body'],
 'title': 'send_email_tool_args',
 'type': 'object',
 'additionalProperties': False}

In [13]:
INSTRUCTIONS4 = """
You are provided with a detailed report. Use your tool to send an email, converting the report into
a clean, well presented HTML email with an appropriate subject line.
"""

email_agent = Agent(name="Email Agent", instructions=INSTRUCTIONS4, tools=[send_email_tool], model=MODEL_NAME)

## Now Orchestrate by LLM

In [17]:
tool1 = planner_agent.as_tool(tool_name="planner_agent", tool_description="use this tool to plan the research based on the query provided")
tool2 = search_agent.as_tool(tool_name="search_agent", tool_description="use this tool to execute the search queries provided")
tool3 = writer_agent.as_tool(tool_name="writer_agent", tool_description="use this tool to write the content based on the input provided")
tool4 = email_agent.as_tool(tool_name="send_email_agent",tool_description="use this tool to send out an email with the given subject and body to all sales prospects")

In [18]:
tools = [tool1,tool2,tool3,tool4]

In [20]:
manager_instructions = "you are a manager which manages the deep research. You will be provided with multiple tools and you have to make use of them efficiently"

manager_task = """Follow this steps :
            1.Based on the Provided query, you should utilise your "planner_agent" to plan all the queries which needs to be searched.
            2.Once you get all the plan for the research, use your "search_agent" to execute all the search queries.
            3.Output from the search queries needs to be provided to the "writer_agent", which will draft a beautiful content about the research done.
            4.Finally, the content generated needs to be sent on email with the help of "send_email_agent"

        Your Query is : Most Popular AI agent Framework in 2026

"""

In [21]:
manager = Agent(name="research_manager",instructions=manager_instructions,model=MODEL_NAME,tools=tools)

In [22]:
with trace("Sales manager"):
    result = await Runner.run(manager,manager_task)